# IVFC Station 187 — Staffing Prediction & Rhythmic Alarm Detection\n",
    "\n",
    "**Two models in one notebook:**\n",
    "\n",
    "1. **LSTM Staffing Predictor** — trained on the full historical incident record (2010–present) to forecast how many firefighters will be needed tomorrow and the next 7 days.\n",
    "\n",
    "2. **Rhythmic Alarm Detector** — flags addresses that generate calls on suspiciously regular intervals, indicating faulty equipment or habitual false alarms.\n",
    "\n",
    "---\n",
    "**Primary data source:** `incidents_historical.csv` — NFIRS-coded incident log, 2010–present  \n",
    "**Rolling data source:** `D90.csv` — last 90 days from First Due SizeUp scraper  \n",
    "The notebook automatically merges both when both are available.

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────
import subprocess, sys

for pkg in ["tensorflow", "scikit-learn", "matplotlib", "pandas",
            "numpy", "google-cloud-storage"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("Dependencies ready.")

In [ ]:
import os, re, math, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings("ignore")

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error

tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow {tf.__version__}")

## Section 1 — Configuration

In [ ]:
# ── Data sources ─────────────────────────────────────────────────────────
# Primary: full historical NFIRS export (2010-present)
HISTORICAL_CSV  = "incidents_historical.csv"

# Rolling: last 90 days from First Due scraper (merged in if available)
GCS_BUCKET_NAME = os.environ.get("GCS_BUCKET_NAME", "ivfc-reports-187")
GCS_CSV_PATH    = "data/D90.csv"
LOCAL_D90_PATH  = "D90.csv"

# Station filter — set to None to include all stations
STATION_FILTER  = "187"

# ── LSTM hyperparameters ─────────────────────────────────────────────────
WINDOW_DAYS   = 30    # days of history per sequence (more data → longer window OK)
FORECAST_DAYS = 7
EPOCHS        = 200
BATCH_SIZE    = 32
LSTM_UNITS_1  = 64
LSTM_UNITS_2  = 32
DROPOUT       = 0.2

# ── Staffing rules ───────────────────────────────────────────────────────
MIN_STAFF = 2
P_REC     = 0.90
P_SURGE   = 0.95

# ── Rhythmic alarm thresholds ────────────────────────────────────────────
MIN_CALLS_FOR_ANALYSIS = 5     # more data → raise this threshold
RHYTHMIC_CV_THRESHOLD  = 0.50

print("Configuration loaded.")

## Section 2 — Load Dispatch CSV

In [ ]:
def load_from_gcs(bucket_name, blob_name, local_dest):
    try:
        from google.cloud import storage
        client = storage.Client()
        bucket = client.bucket(bucket_name)
        bucket.blob(blob_name).download_to_filename(local_dest)
        print(f"Downloaded gs://{bucket_name}/{blob_name}")
        return local_dest
    except Exception as e:
        print(f"GCS unavailable ({e})")
        return None

# ── NFIRS incident type → crew size ──────────────────────────────────────
# Codes: https://www.nfirs.fema.gov/documentation/reference/
NFIRS_CREW = [
    # Structure fires — large crew
    (re.compile(r"^1[01][0-9]", re.I), 8),   # 111-119 building fires
    (re.compile(r"^12[0-9]", re.I),    6),    # 120s outside fires
    # Vehicle / wildland
    (re.compile(r"^13[0-9]", re.I),    5),    # 130s mobile property fires
    (re.compile(r"^14[0-9]", re.I),    4),    # 140s natural vegetation
    # Rescue / EMS
    (re.compile(r"^35[0-9]", re.I),    6),    # 350s extrication
    (re.compile(r"^32[0-9]", re.I),    4),    # 320s motor vehicle accident
    (re.compile(r"^3[0-9][0-9]", re.I), 3),  # other rescue
    # Hazmat
    (re.compile(r"^4[0-9][0-9]", re.I), 4),  # 400s hazardous conditions
    # False alarms / cancelled
    (re.compile(r"^6[0-9][0-9]", re.I), 2),  # 600s good intent / cancelled
    (re.compile(r"^7[0-9][0-9]", re.I), 2),  # 700s false alarms
    # Service calls
    (re.compile(r"^5[0-9][0-9]", re.I), 2),  # 500s service calls
]

# ── NFIRS incident type → duration (minutes) ─────────────────────────────
NFIRS_DUR = [
    (re.compile(r"^1[01][0-9]", re.I), 120),  # structure fires
    (re.compile(r"^12[0-9]", re.I),     60),
    (re.compile(r"^13[0-9]", re.I),     75),  # vehicle fires
    (re.compile(r"^14[0-9]", re.I),     90),  # wildland
    (re.compile(r"^35[0-9]", re.I),     90),  # extrication
    (re.compile(r"^32[0-9]", re.I),     60),  # MVA
    (re.compile(r"^3[0-9][0-9]", re.I), 45),
    (re.compile(r"^4[0-9][0-9]", re.I), 60),
    (re.compile(r"^6[0-9][0-9]", re.I), 20),  # cancelled
    (re.compile(r"^7[0-9][0-9]", re.I), 30),  # false alarms
    (re.compile(r"^5[0-9][0-9]", re.I), 30),
]

# ── First Due free-text → crew / duration (D90 format) ───────────────────
FIRSTDUE_CREW = [
    (re.compile(r"POSSIBLE RES BLDG FIRE|WORKING FIRE|STRUCTURE|BLDG FIRE", re.I), 8),
    (re.compile(r"VEHICLE FIRE", re.I), 5),
    (re.compile(r"EXTRICATION|ENTRAP|PINNED|TRAPPED", re.I), 6),
    (re.compile(r"CRASH|MVC|MVA|COLLISION|ACCIDENT", re.I), 4),
    (re.compile(r"FIRE ALARM|ALARM", re.I), 3),
    (re.compile(r"MEDICAL|EMS|ASSIST", re.I), 2),
]
FIRSTDUE_DUR = [
    (re.compile(r"POSSIBLE RES BLDG FIRE|WORKING FIRE|STRUCTURE|BLDG FIRE", re.I), 120),
    (re.compile(r"VEHICLE FIRE", re.I), 75),
    (re.compile(r"EXTRICATION|ENTRAP|PINNED|TRAPPED", re.I), 90),
    (re.compile(r"CRASH|MVC|MVA|COLLISION", re.I), 60),
    (re.compile(r"FIRE ALARM|ALARM", re.I), 45),
    (re.compile(r"MEDICAL|EMS", re.I), 30),
]

DEFAULT_CREW = 3
DEFAULT_DUR  = 45

def extract_nfirs_code(t):
    """Pull the leading numeric code from NFIRS description like '745 - Alarm system...'"""
    if not isinstance(t, str): return None
    m = re.match(r"^\s*(\d{3})", t.strip())
    return m.group(1) if m else None

def crew_for(t, nfirs=True):
    if not isinstance(t, str): return DEFAULT_CREW
    rules = NFIRS_CREW if nfirs else FIRSTDUE_CREW
    code  = extract_nfirs_code(t) if nfirs else None
    check = code if code else t
    for pat, c in rules:
        if pat.search(check): return c
    return DEFAULT_CREW

def duration_for(t, nfirs=True):
    if not isinstance(t, str): return DEFAULT_DUR
    rules = NFIRS_DUR if nfirs else FIRSTDUE_DUR
    code  = extract_nfirs_code(t) if nfirs else None
    check = code if code else t
    for pat, d in rules:
        if pat.search(check): return d
    return DEFAULT_DUR

# ── Load historical NFIRS CSV ─────────────────────────────────────────────
hist_raw = pd.read_csv(HISTORICAL_CSV, low_memory=False)
hist_raw.columns = [c.strip() for c in hist_raw.columns]

# Filter to Station 187 if column exists
if STATION_FILTER and "StationNumber" in hist_raw.columns:
    hist_raw = hist_raw[hist_raw["StationNumber"].astype(str).str.contains(STATION_FILTER, na=False)]
    print(f"Filtered to station {STATION_FILTER}: {len(hist_raw):,} rows")

# Build datetime from split date + time columns
hist_raw["start_dt"] = pd.to_datetime(
    hist_raw["IncidentDate"].astype(str) + " " + hist_raw["IncidentTime"].astype(str),
    errors="coerce"
)
hist_raw["call_type"]   = hist_raw["IncidentType"].fillna("") if "IncidentType" in hist_raw.columns else ""
hist_raw["address_raw"] = hist_raw["Address"].fillna("")      if "Address"      in hist_raw.columns else ""
hist_raw["crew"]        = hist_raw["call_type"].apply(lambda t: crew_for(t, nfirs=True))
hist_raw["duration_min"]= hist_raw["call_type"].apply(lambda t: duration_for(t, nfirs=True))
hist_raw["end_dt"]      = hist_raw["start_dt"] + pd.to_timedelta(hist_raw["duration_min"], unit="m")
hist_raw["source"]      = "nfirs"

hist = hist_raw.dropna(subset=["start_dt"]).copy()
hist = hist[hist["end_dt"] >= hist["start_dt"]].copy()
print(f"Historical (NFIRS): {len(hist):,} incidents")
print(f"  Date range: {hist['start_dt'].min().date()} → {hist['start_dt'].max().date()}")

# ── Load D90 and merge (optional) ────────────────────────────────────────
import tempfile as _tmp
_gcs_dest = os.path.join(_tmp.gettempdir(), "sp_D90.csv")
_dl = load_from_gcs(GCS_BUCKET_NAME, GCS_CSV_PATH, _gcs_dest)
d90_path = _dl if _dl else (LOCAL_D90_PATH if os.path.exists(LOCAL_D90_PATH) else None)

d90_incidents = None
if d90_path:
    try:
        d90_raw = pd.read_csv(d90_path)
        d90_raw.columns = [c.strip() for c in d90_raw.columns]
        date_col = next((c for c in d90_raw.columns if "DATE" in c.upper() or "TIME" in c.upper()), None)
        type_col = next((c for c in d90_raw.columns if "TYPE" in c.upper()), None)
        addr_col = next((c for c in d90_raw.columns if "ADDRESS" in c.upper()), None)
        if date_col:
            d90_raw["start_dt"]    = pd.to_datetime(d90_raw[date_col], errors="coerce")
            d90_raw["call_type"]   = d90_raw[type_col].fillna("") if type_col else ""
            d90_raw["address_raw"] = d90_raw[addr_col].fillna("") if addr_col else ""
            d90_raw["crew"]        = d90_raw["call_type"].apply(lambda t: crew_for(t, nfirs=False))
            d90_raw["duration_min"]= d90_raw["call_type"].apply(lambda t: duration_for(t, nfirs=False))
            d90_raw["end_dt"]      = d90_raw["start_dt"] + pd.to_timedelta(d90_raw["duration_min"], unit="m")
            d90_raw["source"]      = "firstdue_d90"
            d90_incidents = d90_raw.dropna(subset=["start_dt"]).copy()
            # Only keep rows newer than what's in historical to avoid overlap
            hist_cutoff = hist["start_dt"].max()
            d90_incidents = d90_incidents[d90_incidents["start_dt"] > hist_cutoff].copy()
            print(f"D90 (First Due, new rows only): {len(d90_incidents):,} incidents")
    except Exception as e:
        print(f"D90 load failed: {e}")

# ── Merge ─────────────────────────────────────────────────────────────────
SHARED_COLS = ["start_dt", "end_dt", "crew", "duration_min", "call_type", "address_raw", "source"]
frames = [hist[SHARED_COLS]]
if d90_incidents is not None and len(d90_incidents) > 0:
    frames.append(d90_incidents[SHARED_COLS])

df = pd.concat(frames, ignore_index=True).sort_values("start_dt").reset_index(drop=True)
print(f"\nCombined dataset: {len(df):,} incidents")
print(f"Date range: {df['start_dt'].min().date()} → {df['start_dt'].max().date()}")

## Section 3 — Feature Engineering

In [ ]:
# ── Validate merged dataset ───────────────────────────────────────────────
print("═" * 55)
print("  MERGED DATASET SUMMARY")
print("═" * 55)
print(f"  Total incidents : {len(df):,}")
print(f"  Date range      : {df['start_dt'].min().date()} → {df['start_dt'].max().date()}")
print(f"  Sources         : {df['source'].value_counts().to_dict()}")
print(f"  Unique addresses: {df['address_raw'].nunique():,}")
print(f"  Avg crew / call : {df['crew'].mean():.2f}")
print(f"  Avg duration    : {df['duration_min'].mean():.1f} min")
print()
print("Columns:", list(df.columns))
print()
print("Sample rows:")
df.head(5)

In [ ]:
# ── Call type distribution ────────────────────────────────────────────────
# Top 15 incident types across the combined dataset
type_counts = df["call_type"].value_counts().head(15)
print("Top 15 call types:")
print(type_counts.to_string())

In [ ]:
# ── Build daily staffing table ────────────────────────────────────────────
def daily_staffing(incidents, resolution_min=5, p_rec=P_REC, p_surge=P_SURGE, min_staff=MIN_STAFF):
    if incidents.empty:
        return pd.DataFrame(columns=["date","incident_count","baseline","recommended","surge","staff_for_day"])

    step = pd.Timedelta(minutes=resolution_min)
    start_date = incidents["start_dt"].min().floor("D")
    end_date   = incidents["end_dt"].max().ceil("D")
    days = pd.date_range(start_date, end_date, freq="D", inclusive="left")
    inc = incidents.sort_values("start_dt").reset_index(drop=True)
    rows = []

    for day in days:
        day_end = day + pd.Timedelta(days=1)
        mask    = (inc["start_dt"] < day_end) & (inc["end_dt"] > day)
        day_inc = inc.loc[mask]
        n_calls = int((inc["start_dt"] >= day).sum() & True) if not day_inc.empty else 0
        # count calls that START on this day
        n_calls = int(((inc["start_dt"] >= day) & (inc["start_dt"] < day_end)).sum())

        if day_inc.empty:
            rows.append({"date": day.date(), "incident_count": 0,
                         "baseline": min_staff, "recommended": min_staff,
                         "surge": min_staff, "staff_for_day": min_staff})
            continue

        ticks  = pd.date_range(day, day_end, freq=step, inclusive="left").values
        starts = day_inc["start_dt"].values
        ends   = day_inc["end_dt"].values
        crews  = day_inc["crew"].values
        needed = [int(crews[(starts <= t) & (ends > t)].sum()) for t in ticks]
        s = pd.Series(needed)

        rows.append({
            "date":          day.date(),
            "incident_count": n_calls,
            "baseline":      max(min_staff, math.ceil(s.mean())),
            "recommended":   max(min_staff, math.ceil(s.quantile(p_rec))),
            "surge":         max(min_staff, math.ceil(s.quantile(p_surge))),
            "staff_for_day": max(min_staff, math.ceil(s.quantile(p_rec))),
        })

    return pd.DataFrame(rows)

daily = daily_staffing(df)
daily["date"] = pd.to_datetime(daily["date"])
print(f"Daily table: {len(daily)} rows")
daily.tail(10)

In [ ]:
# ── Quick exploratory chart ───────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.bar(daily["date"], daily["incident_count"], color="#2980B9", alpha=0.7, width=0.8)
ax1.set_ylabel("Calls per Day")
ax1.set_title("Station 187 — Daily Incident Volume", fontweight="bold")
ax1.grid(axis="y", alpha=0.3)

ax2.fill_between(daily["date"], daily["baseline"], daily["surge"],
                 alpha=0.15, color="#C0392B", label="Baseline–Surge band")
ax2.plot(daily["date"], daily["staff_for_day"], color="#C0392B",
         linewidth=1.5, label="Recommended staff")
ax2.set_ylabel("Firefighters")
ax2.set_title("Daily Recommended Staffing (90th pct concurrent crew)", fontweight="bold")
ax2.legend(fontsize=9)
ax2.grid(axis="y", alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax2.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
plt.xticks(rotation=40, ha="right", fontsize=8)
plt.tight_layout()
plt.savefig("/tmp/staffing_history.png", dpi=130, bbox_inches="tight")
plt.show()
print("Chart saved.")

---
## Section 4 — LSTM Staffing Prediction

**Architecture:** Two stacked LSTM layers → Dense head → single output (tomorrow's recommended staff count)

**Input features per day:**
- `incident_count` — raw call volume  
- `staff_for_day` — concurrent crew demand (target variable, also used as input for context)
- `day_sin / day_cos` — cyclical day-of-week encoding  
- `month_sin / month_cos` — cyclical month encoding  
- `is_weekend` — binary flag  

**Note on data size:** With D90 (~90 days) and a 14-day window, training has ~76 sequences — enough to learn basic patterns. Predictions will improve substantially as the full historical dataset is loaded.

In [ ]:
# ── Build feature matrix ──────────────────────────────────────────────────
feat = daily.copy().sort_values("date").reset_index(drop=True)

feat["dow"]        = feat["date"].dt.dayofweek          # 0=Mon … 6=Sun
feat["month"]      = feat["date"].dt.month
feat["day_sin"]    = np.sin(2 * np.pi * feat["dow"] / 7)
feat["day_cos"]    = np.cos(2 * np.pi * feat["dow"] / 7)
feat["month_sin"]  = np.sin(2 * np.pi * feat["month"] / 12)
feat["month_cos"]  = np.cos(2 * np.pi * feat["month"] / 12)
feat["is_weekend"] = (feat["dow"] >= 5).astype(int)

FEATURE_COLS = ["incident_count", "staff_for_day",
                "day_sin", "day_cos", "month_sin", "month_cos", "is_weekend"]
TARGET_COL   = "staff_for_day"

X_raw = feat[FEATURE_COLS].values.astype(np.float32)
y_raw = feat[TARGET_COL].values.astype(np.float32)

n_days    = len(feat)
n_seq     = n_days - WINDOW_DAYS
n_features = len(FEATURE_COLS)

print(f"Total days:   {n_days}")
print(f"Window:       {WINDOW_DAYS} days")
print(f"Sequences:    {n_seq}")
print(f"Features:     {FEATURE_COLS}")

if n_seq < 20:
    print(f"\n⚠️  Only {n_seq} training sequences — model will train but predictions"
          " will be approximate. Load full historical CSV for better accuracy.")

In [ ]:
# ── Scale features ────────────────────────────────────────────────────────
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X_raw)
y_scaled = scaler_y.fit_transform(y_raw.reshape(-1, 1)).flatten()

# ── Build sliding-window sequences ───────────────────────────────────────
X_seq, y_seq = [], []
for i in range(n_seq):
    X_seq.append(X_scaled[i : i + WINDOW_DAYS])
    y_seq.append(y_scaled[i + WINDOW_DAYS])

X_seq = np.array(X_seq)   # (n_seq, WINDOW_DAYS, n_features)
y_seq = np.array(y_seq)   # (n_seq,)

# ── Train / val split (80 / 20, time-ordered) ────────────────────────────
split = int(len(X_seq) * 0.80)
X_train, X_val = X_seq[:split], X_seq[split:]
y_train, y_val = y_seq[:split], y_seq[split:]

print(f"Train: {len(X_train)} sequences | Val: {len(X_val)} sequences")
print(f"Input shape: {X_seq.shape}")

In [ ]:
# ── Build LSTM model ──────────────────────────────────────────────────────
model = Sequential([
    Input(shape=(WINDOW_DAYS, n_features)),
    LSTM(LSTM_UNITS_1, return_sequences=True),
    Dropout(DROPOUT),
    LSTM(LSTM_UNITS_2, return_sequences=False),
    Dropout(DROPOUT),
    Dense(16, activation="relu"),
    Dense(1),
])

model.compile(optimizer="adam", loss="mae", metrics=["mse"])
model.summary()

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────
early_stop = EarlyStopping(
    monitor="val_loss", patience=20, restore_best_weights=True, verbose=1
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=0,
)

print(f"Stopped at epoch {len(history.history['loss'])}")
print(f"Best val MAE: {min(history.history['val_loss']):.4f} (scaled)")

In [ ]:
# ── Evaluate on validation set ────────────────────────────────────────────
y_val_pred_scaled = model.predict(X_val, verbose=0).flatten()
y_val_pred = scaler_y.inverse_transform(y_val_pred_scaled.reshape(-1, 1)).flatten()
y_val_true = scaler_y.inverse_transform(y_val.reshape(-1, 1)).flatten()

mae = mean_absolute_error(y_val_true, y_val_pred)
print(f"Validation MAE: {mae:.2f} firefighters")

# ── Plot training loss ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history.history["loss"],     label="Train loss", color="#2980B9")
axes[0].plot(history.history["val_loss"], label="Val loss",   color="#C0392B")
axes[0].set_title("Training Loss (MAE)", fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)

val_dates = feat["date"].values[split + WINDOW_DAYS:]
axes[1].plot(val_dates, y_val_true, "o-", label="Actual",    color="#2C3E50", linewidth=1.5)
axes[1].plot(val_dates, y_val_pred, "s--", label="Predicted", color="#C0392B", linewidth=1.5)
axes[1].set_title(f"Validation: Actual vs Predicted (MAE = {mae:.2f})", fontweight="bold")
axes[1].set_ylabel("Recommended Staff")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30, ha="right")

plt.tight_layout()
plt.savefig("/tmp/lstm_eval.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
# ── Forecast next FORECAST_DAYS days ─────────────────────────────────────
# Seed: last WINDOW_DAYS days of known data
seed_window = X_scaled[-WINDOW_DAYS:].copy()   # (WINDOW_DAYS, n_features)

last_date = feat["date"].iloc[-1]
forecast_rows = []

for i in range(1, FORECAST_DAYS + 1):
    inp = seed_window[-WINDOW_DAYS:].reshape(1, WINDOW_DAYS, n_features)
    pred_scaled = model.predict(inp, verbose=0)[0, 0]
    pred_staff  = float(scaler_y.inverse_transform([[pred_scaled]])[0, 0])
    pred_staff  = max(MIN_STAFF, round(pred_staff))

    fdate = last_date + pd.Timedelta(days=i)
    dow   = fdate.dayofweek
    mon   = fdate.month

    # Build next row's features using predicted staff as 'staff_for_day'
    # incident_count estimated as rolling 7-day average
    est_calls = feat["incident_count"].tail(7).mean()
    next_raw  = np.array([[est_calls, pred_staff,
                            np.sin(2*np.pi*dow/7), np.cos(2*np.pi*dow/7),
                            np.sin(2*np.pi*mon/12), np.cos(2*np.pi*mon/12),
                            int(dow >= 5)]], dtype=np.float32)
    next_scaled = scaler_X.transform(next_raw)[0]
    seed_window = np.vstack([seed_window, next_scaled])

    forecast_rows.append({
        "date":             fdate,
        "day_of_week":      fdate.strftime("%A"),
        "predicted_staff":  pred_staff,
        "est_calls":        round(est_calls, 1),
    })

forecast_df = pd.DataFrame(forecast_rows)
print("\n═══════════════════════════════════════")
print("  LSTM STAFFING FORECAST")
print("═══════════════════════════════════════")
print(forecast_df.to_string(index=False))

In [ ]:
# ── Forecast chart ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

# Historical (last 30 days)
hist_tail = daily.tail(30)
ax.plot(hist_tail["date"], hist_tail["staff_for_day"],
        "o-", color="#2C3E50", linewidth=1.5, label="Historical (recommended)")

# Forecast
ax.bar(forecast_df["date"], forecast_df["predicted_staff"],
       color="#E74C3C", alpha=0.75, width=0.7, label="LSTM forecast")

# Divider
ax.axvline(last_date, color="gray", linestyle="--", linewidth=1, alpha=0.7)
ax.text(last_date, ax.get_ylim()[1] * 0.95, "  Forecast →",
        fontsize=9, color="gray", va="top")

ax.set_title(f"Station 187 — Staffing Forecast (Next {FORECAST_DAYS} Days)",
             fontweight="bold", fontsize=13)
ax.set_ylabel("Recommended Firefighters")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(bottom=0)
plt.xticks(rotation=35, ha="right", fontsize=9)
plt.tight_layout()
plt.savefig("/tmp/lstm_forecast.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
# ── Save model + forecast ─────────────────────────────────────────────────
model.save("/tmp/ivfc_staffing_lstm.keras")
forecast_df.to_csv("staffing_forecast.csv", index=False)
daily.to_csv("daily_staffing.csv", index=False)
print("Model saved: /tmp/ivfc_staffing_lstm.keras")
print("Forecast saved: staffing_forecast.csv")
print("Daily history saved: daily_staffing.csv")

---
## Section 5 — Rhythmic Alarm Detection

Addresses that generate calls on suspiciously **regular intervals** are likely faulty equipment or nuisance alarms. 

**Method:**  
For each address with ≥ 3 calls, compute the **coefficient of variation (CV)** of the gaps between calls:  
`CV = std(gaps) / mean(gaps)`  

- **CV < 0.5** → calls are evenly spaced → flag as rhythmic  
- **CV ≥ 0.5** → irregular spacing → likely random / genuine incidents

Also reports the most common **time of day** and **day of week** for repeat callers.

In [ ]:
# ── Normalize addresses ───────────────────────────────────────────────────
def normalize_address(s):
    if not isinstance(s, str) or not s.strip():
        return None
    s = s.upper().strip()
    # Remove unit/apt/suite suffixes
    s = re.sub(r"\b(APT|UNIT|STE|SUITE|FL|FLOOR|RM|ROOM)[\s#]*[\w-]*", "", s)
    # Normalize common street abbreviations
    abbr = {"STREET": "ST", "AVENUE": "AVE", "BOULEVARD": "BLVD",
            "DRIVE": "DR", "ROAD": "RD", "LANE": "LN",
            "COURT": "CT", "CIRCLE": "CIR", "PLACE": "PL"}
    for full, short in abbr.items():
        s = re.sub(rf"\b{full}\b", short, s)
    # Collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()
    # Drop city/state/zip (keep street address only)
    s = re.sub(r",?\s*(PITTSBURGH|WEXFORD|MCCANDLESS|PA|\d{5}).*$", "", s).strip()
    return s if s else None

df["address"] = df["address_raw"].apply(normalize_address)
addr_df = df[["address", "start_dt", "call_type"]].dropna(subset=["address"]).copy()

n_unique = addr_df["address"].nunique()
print(f"{len(addr_df):,} calls with valid addresses across {n_unique:,} unique locations")

In [ ]:
# ── Compute inter-arrival stats per address ───────────────────────────────
results = []

for addr, grp in addr_df.groupby("address"):
    grp = grp.sort_values("start_dt")
    n   = len(grp)
    if n < MIN_CALLS_FOR_ANALYSIS:
        continue

    gaps_hrs = grp["start_dt"].diff().dt.total_seconds().dropna() / 3600

    mean_gap = gaps_hrs.mean()
    std_gap  = gaps_hrs.std()
    cv       = (std_gap / mean_gap) if mean_gap > 0 else np.nan
    min_gap  = gaps_hrs.min()

    # Time-of-day pattern (most common 2-hour block)
    tod_block = (grp["start_dt"].dt.hour // 2 * 2).mode()
    tod_str   = f"{tod_block.iloc[0]:02d}:00–{tod_block.iloc[0]+2:02d}:00" if len(tod_block) else "?"

    # Most common day of week
    dow_mode = grp["start_dt"].dt.day_name().mode()
    dow_str  = dow_mode.iloc[0] if len(dow_mode) else "?"

    # Most common call type
    top_type = grp["call_type"].mode().iloc[0] if not grp["call_type"].mode().empty else "?"

    # Span of calls
    span_days = (grp["start_dt"].max() - grp["start_dt"].min()).days

    results.append({
        "address":         addr,
        "call_count":      n,
        "span_days":       span_days,
        "mean_gap_hrs":    round(mean_gap, 1),
        "std_gap_hrs":     round(std_gap,  1),
        "cv":              round(cv,       3) if not np.isnan(cv) else None,
        "min_gap_hrs":     round(min_gap,  1),
        "common_time":     tod_str,
        "common_dow":      dow_str,
        "top_call_type":   top_type,
        "rhythmic_flag":   (cv is not None and not np.isnan(cv) and cv < RHYTHMIC_CV_THRESHOLD),
    })

alarm_df = pd.DataFrame(results).sort_values(["rhythmic_flag", "call_count"], ascending=[False, False])
print(f"Analyzed {len(alarm_df)} addresses with ≥{MIN_CALLS_FOR_ANALYSIS} calls")
print(f"Rhythmic flags: {alarm_df['rhythmic_flag'].sum()}")

In [ ]:
# ── Print flagged addresses ───────────────────────────────────────────────
flagged = alarm_df[alarm_df["rhythmic_flag"]].copy()

if flagged.empty:
    print("No rhythmic alarms detected in this dataset.")
    print("(May appear with more data — D90 is a small window.)")
else:
    print("\n╔══════════════════════════════════════════════════════════╗")
    print("║       RHYTHMIC / FAULTY ALARM CANDIDATES                 ║")
    print("╚══════════════════════════════════════════════════════════╝")
    for _, row in flagged.iterrows():
        print(f"\n  📍 {row['address']}")
        print(f"     Calls: {row['call_count']}  |  Span: {row['span_days']} days")
        print(f"     Avg gap: {row['mean_gap_hrs']} hrs  |  CV: {row['cv']} (threshold < {RHYTHMIC_CV_THRESHOLD})")
        print(f"     Min gap: {row['min_gap_hrs']} hrs")
        print(f"     Most common: {row['common_dow']} @ {row['common_time']}")
        print(f"     Top call type: {row['top_call_type']}")

In [ ]:
# ── Top repeat addresses chart (flagged + high-frequency) ─────────────────
top_n = alarm_df.head(12).copy()

if top_n.empty:
    print("Not enough repeat addresses to chart.")
else:
    colors = ["#C0392B" if f else "#2980B9" for f in top_n["rhythmic_flag"]]
    labels = [a[:35] + "…" if len(a) > 35 else a for a in top_n["address"]]

    fig, ax = plt.subplots(figsize=(11, 6))
    bars = ax.barh(range(len(top_n)), top_n["call_count"], color=colors, edgecolor="white")

    for bar, cv in zip(bars, top_n["cv"]):
        cv_txt = f"  CV={cv}" if cv is not None else ""
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                cv_txt, va="center", fontsize=8, color="#555")

    ax.set_yticks(range(len(top_n)))
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel("Number of Calls")
    ax.set_title("Top Repeat Addresses — Red = Rhythmic Flag (CV < 0.5)",
                 fontweight="bold")
    ax.grid(axis="x", alpha=0.3)

    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(facecolor="#C0392B", label="Rhythmic / Faulty alarm candidate"),
        Patch(facecolor="#2980B9", label="Repeat but irregular"),
    ], loc="lower right", fontsize=9)

    plt.tight_layout()
    plt.savefig("/tmp/rhythmic_alarms.png", dpi=130, bbox_inches="tight")
    plt.show()

In [ ]:
# ── Timeline of calls for flagged addresses ───────────────────────────────
if not flagged.empty:
    fig, ax = plt.subplots(figsize=(13, max(3, len(flagged) * 0.9)))

    for i, (_, row) in enumerate(flagged.iterrows()):
        addr  = row["address"]
        times = addr_df[addr_df["address"] == addr]["start_dt"].sort_values()
        label = addr[:40] + "…" if len(addr) > 40 else addr
        ax.scatter(times, [i] * len(times), s=80, color="#C0392B",
                   zorder=3, label=label)
        ax.hlines(i, times.min(), times.max(),
                  color="#C0392B", alpha=0.25, linewidth=1)

    ax.set_yticks(range(len(flagged)))
    ax.set_yticklabels([r["address"][:40] for _, r in flagged.iterrows()], fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
    plt.xticks(rotation=35, ha="right", fontsize=8)
    ax.set_title("Call Timeline — Rhythmic Alarm Candidates",
                 fontweight="bold")
    ax.grid(axis="x", alpha=0.2)
    plt.tight_layout()
    plt.savefig("/tmp/rhythmic_timeline.png", dpi=130, bbox_inches="tight")
    plt.show()
else:
    print("No flagged addresses to plot timeline for.")

In [ ]:
# ── Save full address analysis ────────────────────────────────────────────
alarm_df.to_csv("rhythmic_alarm_analysis.csv", index=False)
print(f"Saved: rhythmic_alarm_analysis.csv ({len(alarm_df)} addresses)")

print("\n══════════════════════════════════════════")
print("  ALL ADDRESSES (≥3 calls, sorted by CV)")
print("══════════════════════════════════════════")
display_cols = ["address", "call_count", "mean_gap_hrs", "cv", "common_dow",
                "common_time", "top_call_type", "rhythmic_flag"]
print(alarm_df[display_cols].to_string(index=False))